# AquaLens — Underwater Marine Species and Coral Reef Health Visual Assessment
### End-to-End Deep Learning & Optical Enhancement Pipeline
---
**Project:** AquaLens  
**Platform:** Google Colab (Free T4 GPU or CPU fallback)  
**Key Capabilities:**
1. **Custom Optical Preprocessing:** OpenCV/NumPy Dark Channel Prior (DCP) color correction + Lightness CLAHE
2. **Transfer Learning:** Fine-tuned ResNet-34 with Progressive Unfreezing schedule
3. **Dual Vision Tasks:** (A) Fish Species Classification, (B) Coral Bleaching Detection
4. **Visual Explainability:** Grad-CAM saliency heatmaps overlaid on underwater scenes
5. **Ecological Health Index:** Heuristic bleaching severity scoring and Reef Health Index (RHI)
6. **Rigorous Ablation Study:** 4-way empirical comparison across Raw, CLAHE, DCP, and DCP+CLAHE

## 1. System Environment & Hardware Verification
Verifying GPU acceleration (e.g., NVIDIA T4) or falling back to CPU mode.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running in CPU mode. Pipeline will use CPU-friendly resolution and epoch limits.")

## 2. Repository Setup & Dependencies

In [ ]:
# Clone repository if running in fresh Colab session
import os, sys
if not os.path.exists("AquaLens") and not os.path.exists("src"):
    !git clone https://github.com/your-username/AquaLens.git
    %cd AquaLens

!pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())
print("Environment configured successfully.")

## 3. Dataset Setup (Kaggle or Synthetic Placeholder)
Provide your Kaggle credentials below. If omitted, the notebook automatically generates a synthetic placeholder dataset so the entire pipeline can execute end-to-end without interruption.

In [ ]:
import os
# Optional: Uncomment and enter your credentials if you have a Kaggle account:
# os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"
# os.environ["KAGGLE_KEY"] = "your_kaggle_api_key"

!python data/download_data.py --config config.yaml

## 4. Optical Preprocessing Demonstration (DCP + CLAHE)
Comparing Raw underwater imagery with our Dark Channel Prior dehazing and L-channel CLAHE enhancement.

In [ ]:
import cv2, glob
import matplotlib.pyplot as plt
from src.preprocessing import preprocess_image, compute_image_quality_metrics
from src.utils import load_config, get_effective_config

cfg = get_effective_config(load_config("config.yaml"))
sample_files = glob.glob("data/**/*.png", recursive=True)

if sample_files:
    test_path = sample_files[0]
    raw = cv2.imread(test_path)
    
    dcp_only = preprocess_image(raw, cfg, apply_dark_channel=True, apply_clahe_flag=False, target_size=224)
    clahe_only = preprocess_image(raw, cfg, apply_dark_channel=False, apply_clahe_flag=True, target_size=224)
    full_enh = preprocess_image(raw, cfg, apply_dark_channel=True, apply_clahe_flag=True, target_size=224)
    
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    titles = ["Raw Underwater", "DCP Dehazing Only", "CLAHE Only", "DCP + CLAHE (Proposed)"]
    imgs = [raw, dcp_only, clahe_only, full_enh]
    
    for ax, im, t in zip(axes, imgs, titles):
        q = compute_image_quality_metrics(im)
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontweight="bold")
        ax.set_xlabel(f"Entropy: {q['entropy']:.2f} | Contrast: {q['contrast']:.2f} | Color: {q['colorfulness']:.1f}")
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

## 5. Model Training: Progressive Unfreezing on ResNet-34
Training the Coral Health Assessment classifier with progressive unfreezing (Epoch 0: FC only, Epoch 3: Layer4 + FC, Epoch 6+: all layers).

In [ ]:
!python -m src.train --config config.yaml --task coral

## 6. Model Training: Fish Species Classification

In [ ]:
!python -m src.train --config config.yaml --task fish

## 7. Model Evaluation & Performance Metrics
Generates confusion matrices, ROC curves, macro-F1, top-k accuracy, and per-class reports.

In [ ]:
# Evaluate Coral Model
!python -m src.evaluate --config config.yaml --task coral --checkpoint outputs/checkpoints/coral_best.pth

# Evaluate Fish Model
!python -m src.evaluate --config config.yaml --task fish --checkpoint outputs/checkpoints/fish_best.pth

## 8. Preprocessing Ablation Study (4-Way Comparison)
Empirically measuring the impact of (1) Raw, (2) CLAHE Only, (3) DCP Only, and (4) Combined DCP + CLAHE on classification accuracy and optical quality.

In [ ]:
!python -m src.ablation --config config.yaml --task coral

# Display generated ablation figure
from IPython.display import Image, display
display(Image(filename="outputs/figures/coral_ablation_comparison.png"))

## 9. Explainability with Grad-CAM & Error Analysis
Visualizing class-activation maps to explain where the neural network focuses and diagnosing misclassifications.

In [ ]:
!python -m src.run_gradcam_analysis --config config.yaml --task fish --checkpoint outputs/checkpoints/fish_best.pth

display(Image(filename="outputs/figures/error_analysis_grid.png"))

## 10. Batch Inference & Reef Health Dashboard
Process a directory of survey frames and compute the aggregate Reef Health Index (RHI).

In [ ]:
!python -m src.batch_inference \
    --input-dir data/coral/bleached \
    --task coral \
    --checkpoint outputs/checkpoints/coral_best.pth \
    --config config.yaml \
    --output-csv outputs/colab_survey_results.csv \
    --save-visualizations

## 11. Launch Interactive Streamlit Web Application
Launch the Streamlit GUI in Colab using localtunnel or ngrok.

In [ ]:
!pip install -q localtunnel
import urllib
print("Public tunnel password/IP:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Run Streamlit in background
!streamlit run app/streamlit_app.py --server.port 8501 & npx localtunnel --port 8501